In [ ]:
pip install geopandas folium matplotlib pandas shapely

In [ ]:
# ==============================
# Ukraine Nightlights Analysis
# ==============================

# 1) Import packages
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import rasterio
from rasterio.plot import show
import rasterstats

# ------------------------------
# 2) Load Ukraine Oblasts GeoJSON
# ------------------------------
# Replace with path to your local GeoJSON
gdf_oblasts = gpd.read_file("ukraine_oblasts.geojson")
print("Oblasts loaded:", gdf_oblasts.head())

# ------------------------------
# 3) Load VIIRS Nightlights GeoTIFF
# ------------------------------
# Replace with path to your downloaded VIIRS GeoTIFF
raster_path = "viirs_ukraine_2023.tif"
src = rasterio.open(raster_path)
print("Raster CRS:", src.crs)
print("Raster dimensions:", src.width, src.height)

# Ensure CRS match
gdf_oblasts = gdf_oblasts.to_crs(src.crs)

# ------------------------------
# 4) Compute average nightlight per oblast
# ------------------------------
# Zonal statistics
stats = rasterstats.zonal_stats(
    gdf_oblasts,
    raster_path,
    stats=['mean'],
    geojson_out=True
)

# Convert to GeoDataFrame
gdf_stats = gpd.GeoDataFrame.from_features(stats)
print(gdf_stats[['name', 'mean']])

# ------------------------------
# 5) Plot Ukraine oblasts with VIIRS overlay
# ------------------------------
fig, ax = plt.subplots(figsize=(12,12))
show(src, ax=ax, cmap='YlOrRd')  # raster background
gdf_oblasts.boundary.plot(ax=ax, edgecolor='black', linewidth=1)
plt.title("Ukraine Oblasts with VIIRS Nightlights")
plt.axis('off')
plt.show()

# ------------------------------
# 6) Add lat/lon points (example cities)
# ------------------------------
points = [
    {'name':'Kyiv','lat':50.45,'lon':30.52},
    {'name':'Lviv','lat':49.84,'lon':24.03},
    {'name':'Odesa','lat':46.48,'lon':30.74},
    {'name':'Kharkiv','lat':50.00,'lon':36.23}
]

df_points = pd.DataFrame(points)
geometry = [Point(xy) for xy in zip(df_points['lon'], df_points['lat'])]
gdf_points = gpd.GeoDataFrame(df_points, geometry=geometry, crs=gdf_oblasts.crs)

# ------------------------------
# 7) Plot final map with cities
# ------------------------------
fig, ax = plt.subplots(figsize=(12,12))
show(src, ax=ax, cmap='YlOrRd')          # VIIRS raster
gdf_oblasts.boundary.plot(ax=ax, edgecolor='black')  # Oblast boundaries
gdf_points.plot(ax=ax, color='blue', markersize=70)  # City points

# Annotate cities
for x, y, label in zip(gdf_points.geometry.x, gdf_points.geometry.y, gdf_points['name']):
    plt.text(x+0.2, y+0.2, label, fontsize=12)

plt.title("Ukraine Oblasts + VIIRS Nightlights + Cities")
plt.axis('off')
plt.show()

# ------------------------------
# 8) Save GeoDataFrame with nightlights stats (optional)
# ------------------------------
gdf_stats.to_file("ukraine_oblasts_viirs_stats.geojson", driver="GeoJSON")

In [ ]:
# ==============================
# Ukraine Nightlights via GEE
# ==============================

# 1) Install / Import packages
# pip install geopandas folium matplotlib pandas shapely geemap earthengine-api
import ee
import geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# 2) Authenticate & Initialize GEE
ee.Authenticate()  # Opens browser for sign-in
ee.Initialize()

# 3) Load Ukraine oblasts GeoJSON
gdf_oblasts = gpd.read_file("ukraine_oblasts.geojson")
gdf_oblasts = gdf_oblasts.to_crs(epsg=4326)  # Ensure WGS84
print(gdf_oblasts.head())

# 4) Convert GeoDataFrame to ee.FeatureCollection
ukraine_fc = geemap.geopandas_to_ee(gdf_oblasts)

# 5) Load VIIRS Nightlights from GEE
viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG') \
    .filterDate('2023-01-01', '2023-12-31') \
    .select('avg_rad')  # Average radiance
viirs_median = viirs.median()  # Reduce to single image

# Clip to Ukraine
viirs_clip = viirs_median.clip(ukraine_fc)

# 6) Compute mean nightlight per oblast (zonal statistics)
mean_viirs = viirs_clip.reduceRegions(
    collection=ukraine_fc,
    reducer=ee.Reducer.mean(),
    scale=500
)

# Convert to Pandas DataFrame
df_viirs = geemap.ee_to_pandas(mean_viirs)
print(df_viirs[['name', 'mean']])

# Merge back into GeoDataFrame for plotting
gdf_oblasts = gdf_oblasts.merge(df_viirs[['name','mean']], on='name', how='left')

# 7) Plot static map with nightlights
ax = gdf_oblasts.plot(column='mean', cmap='YlOrRd', edgecolor='black', legend=True, figsize=(12,10))
plt.title("Ukraine Oblasts - Average VIIRS Nightlights 2023")
plt.axis('off')
plt.show()

# 8) Add lat/lon points (cities)
points = [
    {'name':'Kyiv','lat':50.45,'lon':30.52},
    {'name':'Lviv','lat':49.84,'lon':24.03},
    {'name':'Odesa','lat':46.48,'lon':30.74},
    {'name':'Kharkiv','lat':50.00,'lon':36.23}
]
df_points = pd.DataFrame(points)
geometry = [Point(xy) for xy in zip(df_points['lon'], df_points['lat'])]
gdf_points = gpd.GeoDataFrame(df_points, geometry=geometry, crs='EPSG:4326')

# Plot points on top
ax = gdf_oblasts.plot(column='mean', cmap='YlOrRd', edgecolor='black', legend=True, figsize=(12,10))
gdf_points.plot(ax=ax, color='blue', markersize=70)
for x, y, label in zip(gdf_points.geometry.x, gdf_points.geometry.y, gdf_points['name']):
    plt.text(x+0.2, y+0.2, label, fontsize=12)
plt.title("Ukraine Oblasts + VIIRS Nightlights + Cities")
plt.axis('off')
plt.show()

# 9) Optional interactive map with Folium
m = geemap.Map(center=[48.5,31], zoom=6)
m.add_ee_layer(viirs_clip, {'min':0, 'max':50, 'palette':['black','yellow']}, 'VIIRS Nightlights')
geemap.ee_to_geojson(ukraine_fc, out_file='ukraine_fc.geojson')  # optional save
folium.GeoJson('ukraine_fc.geojson').add_to(m)

# Add city markers
import folium
for idx, row in gdf_points.iterrows():
    folium.Marker([row['lat'], row['lon']], popup=row['name']).add_to(m)

m